In [133]:
%load_ext autoreload
%autoreload 2 

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# AISDI - ćwiczenie 2
## [1] Wprowadzenie
Celem drugiego zadania z przedmiotu Algorytmy i Struktury Danych (AISDI) była implementacja oraz analiza wydajności algorytmów grafowych na przykładzie miejskiej sieci drogowej. Głównym zadaniem było zaimportowanie danych zawierających odległości oraz czasy przejazdu między węzłami w różnych porach dnia, a następnie wyznaczenie optymalnych tras na podstawie kryterium dystansu oraz czasu.

### [1.1] Zaimportowane biblioteki
Zgodnie z wymaganiami projektowymi, do reprezentacji grafu nie wykorzystano gotowych bibliotek (takich jak networkx). Graf został zaimplementowany od podstaw przy użyciu zmodyfikowanej listy sąsiedztwa.

Główną strukturą przechowującą dane jest słownik, którego kluczem jest identyfikator węzła (miasta), a wartością lista obiektów klasy Destination.
Zastosowane narzędzia i moduły:

- csv – do wczytywania i parsowania plików z danymi.

- destination.py – autorska klasa reprezentująca krawędź grafu (cel podróży) wraz z jej wagami (dystans, czas w zależności od pory dnia).

- graph_logic.py – moduł zawierający implementację algorytmów trasowania.

- time – do przeprowadzania pomiarów wydajności i czasu kompilacji algorytmów.

In [134]:
import csv
from pprint import pprint
from destination import Destination
from graph_logic import *
import time

### [1.2] Analiza danych wejściowych
Dane wejściowe pochodzą z pliku city_small.csv. Każdy wiersz reprezentuje krawędź grafu nieskierowanego (dwukierunkową drogę) wraz z zestawem wag. Struktura danych prezentuje się następująco:

In [135]:
with open("data/city_small.csv", newline="") as csv_file:
    small_cities = csv.reader(csv_file)
    for index, row in enumerate(small_cities):
        if index > 5:
            break
        print(row)

['od', 'do', 'odleglosc', 'czas_rano', 'czas_poludnie', 'czas_wieczor']
['0', '9', '17.1', '19', '11', '14']
['0', '20', '6.84', '11', '5', '8']
['1', '11', '9.4', '22', '12', '12']
['1', '21', '8.12', '13', '11', '15']
['1', '22', '19.55', '33', '20', '20']


Konstrukcja grafu odbywa się poprzez iterację po pliku CSV i dwukrotne dodanie krawędzi (od A do B oraz od B do A) do słownika pathways za pomocą metody create_destination().


## [2] Wyznaczanie najkrótszej ścieżki
W pierwszej fazie badań przetestowano działanie klasycznego algorytmu Dijkstry, szukającego najkrótszej ścieżki od wybranego wierzchołka do pozostałych węzłów na podstawie wag odległościowych. Zbadano zachowanie algorytmu w zależności od rozmiaru grafu.

### [2.1] Małe miasto
Algorytm dla małej siatki skrzyżowań odnalazł trasy błyskawicznie. W wynikach prawidłowo zidentyfikowano braki połączeń między odizolowanymi węzłami (oznaczone wartością inf), co świadczy o tym, że badany podgraf nie jest w pełni spójny.

In [136]:

small_pathways = import_city("data/city_small.csv")

start_time = time.perf_counter()
cities_info = get_dijkstra_info_distance(49, small_pathways)
end_time = time.perf_counter()
total_time = end_time - start_time

print_dijkstra_info(cities_info)
print()
print(f"Total time for small city dijkstra is {total_time:.04f}")


Distance from 23 to 32 is 108.20
Distance from 11 to 31 is 72.95
Distance from 1 to 33 is 19.66
Distance from 9 to 49 is inf
Distance from 19 to 48 is 14.28
Distance from 48 to 18 is 18.92
Distance from 10 to 44 is 109.71
Distance from 21 to 5 is 24.83
Distance from 37 to 17 is 92.02
Distance from 32 to 29 is 106.35

Total time for small city dijkstra is 0.0004


### [2.2] Średnie miasto
Dla średniego zbioru danych zauważono oczekiwany wzrost czasu wykonywania algorytmu, związany z większą liczbą krawędzi poddawanych procesowi relaksacji.

In [137]:
mid_pathways = import_city("data/city_mid.csv")

start_time = time.perf_counter()
cities_info = get_dijkstra_info_distance(199, mid_pathways)
end_time = time.perf_counter()

total_time = end_time - start_time

print_dijkstra_info(cities_info)
print()
print(f"Total time for mid city dijkstra is {total_time:.04f}")

Distance from 79 to 144 is 24.82
Distance from 34 to 43 is 76.33
Distance from 169 to 135 is 17.80
Distance from 71 to 184 is 34.55
Distance from 65 to 55 is 51.57
Distance from 142 to 188 is 108.08
Distance from 129 to 159 is 72.67
Distance from 120 to 113 is 38.14
Distance from 148 to 88 is 22.17
Distance from 12 to 169 is 29.25

Total time for mid city dijkstra is 0.0026


### [2.3] Duże miasto 

In [138]:
big_pathways = import_city("data/city_big.csv")

start_time = time.perf_counter()
cities_info = get_dijkstra_info_distance(999, big_pathways)
end_time = time.perf_counter()

total_time = end_time - start_time

print_dijkstra_info(cities_info)
print()
print(f"Total time for big city dijkstra is {total_time:.04f}")

Distance from 717 to 453 is 80.37
Distance from 924 to 706 is 42.81
Distance from 795 to 841 is 27.45
Distance from 135 to 346 is 17.94
Distance from 886 to 340 is 43.68
Distance from 720 to 662 is 61.24
Distance from 351 to 938 is 52.35
Distance from 817 to 974 is 49.98
Distance from 9 to 24 is 86.26
Distance from 828 to 969 is 33.54

Total time for big city dijkstra is 0.0257


### [2.4] Porównanie wydajności
 Czas wyznaczania dystansu skaluje się niezwykle dobrze. Dzięki zastosowaniu kopca dla kolejki priorytetowej, algorytm unika niepotrzebnych iteracji, co pozwala rozwiązywać najkrótszą ścieżkę dla dużych sieci drogowych w zadowalającym, ułamkowym czasie.

## Czas
W tym badaniu zastosowano zmodyfikowaną wersję algorytmu Dijkstry. Jej specyfika polega na śledzeniu upływającego czasu (w minutach od północy) i dynamicznym dobieraniu odpowiedniej wagi (czas rano, w południe lub wieczorem) podczas wjazdu na kolejne skrzyżowanie. Czas początkowy jest ustawiony na 7:30

### [3.1] Małe miasto


In [139]:
start_time = time.perf_counter()
cities_info = get_dijkstra_info_time(49, small_pathways, "07:30")
end_time = time.perf_counter()
total_time = end_time - start_time

print_dijkstra_info(cities_info)
print()
print(f"Total time for small city dijkstra time is {total_time:.04f}")
print(f"This simulation is for starting time equal 07:30")

Arrival time from 13 to 31 is 10:16
Arrival time from 14 to 1 is 09:41
Arrival time from 1 to 35 is inf
Arrival time from 29 to 24 is 09:15
Arrival time from 9 to 46 is inf
Arrival time from 11 to 38 is 09:47
Arrival time from 35 to 14 is inf
Arrival time from 40 to 42 is 08:14
Arrival time from 7 to 35 is inf
Arrival time from 4 to 23 is 08:36

Total time for small city dijkstra time is 0.0005
This simulation is for starting time equal 07:30


### [3.2] Średnie miasto


In [153]:
start_time = time.perf_counter()
cities_info = get_dijkstra_info_time(199, mid_pathways, "07:30")
end_time = time.perf_counter()
total_time = end_time - start_time

print_dijkstra_info(cities_info)
print()
print(f"Total time for  calculating medium city dijkstra time is {total_time:.04f}")
print(f"This simulation is for starting time equal 07:30")

Arrival time from 171 to 18 is 08:45
Arrival time from 11 to 112 is 08:35
Arrival time from 14 to 29 is 08:28
Arrival time from 109 to 129 is 08:33
Arrival time from 115 to 112 is 09:00
Arrival time from 158 to 15 is 07:43
Arrival time from 157 to 1 is 08:17
Arrival time from 171 to 179 is 08:57
Arrival time from 131 to 20 is 08:44
Arrival time from 149 to 162 is 08:30

Total time for  calculating medium city dijkstra time is 0.0101
This simulation is for starting time equal 07:30


### [3.3] Duże miasto

In [ ]:
start_time = time.perf_counter()
cities_info = get_dijkstra_info_time(999, big_pathways, "7:30")
end_time = time.perf_counter()
total_time = end_time - start_time

print_dijkstra_info(cities_info)
print()
print(f"Total time for  calculating big city dijkstra time is {total_time:.04f}")
print(f"This simulation is for starting time equal 07:30")

Arrival time from 717 to 941 is 17:52
Arrival time from 637 to 93 is 17:27
Arrival time from 727 to 615 is 17:06
Arrival time from 782 to 816 is 16:44
Arrival time from 492 to 551 is 17:09
Arrival time from 449 to 574 is 17:22
Arrival time from 756 to 65 is 16:50
Arrival time from 934 to 747 is 17:24
Arrival time from 692 to 481 is 17:42
Arrival time from 298 to 303 is 17:07

Total time for  calculating big city dijkstra time is 0.1171
This simulation is for starting time equal 07:30


### [3.4] Wnioski
Zastosowanie zmodyfikowanej wersji algorytmu Dijkstry
do obliczania czasu przejazdu kończy się pełnym sukcesem.
Czas wykonania (0.06 s) udowadnia, że dynamiczne
pobieranie wag (rano/południe/wieczór) w zależności od
bieżącej godziny nie spowalnia w zauważalny sposób
działania programu dla małych struktur.

## Komiwojazer:
Ten etap był najtrudniejszy do zrealizowania, po pierwsze w wielu przypadkach dotyczących grafu nie mam możliwości przejscia z jednegoe miasta do drugiego bezposrednio, dlatego trzeba zawracać. Dodatkowo niektóre miasta  tworzą graf, który jest odizolowany od głownego grafu, co powoduje, że klaysyczne rozwiązanie problemu kowimojażera staje się niemożliwe, a do tego należy wspomnieć, że tym przypadku została użyta heurystyka, bo dla zwykłego komputeraz obliczenie poprawnej najkrótszej drogi zajmuje n! czasu, dla kilku miast komputer szybko znajdzie najlepszą trasę, ale dla kilkudziesięciu miast potrzebny czas wynosi więcej niż czas istnienie wszechświata. W tym przypadku została użyta najprostrza heurystyka, która nie jest optymalna.


### [4.1] Małe miasto

Tak prezentują się wyniki heurystyki, w której to algortym szuka najkrótszej drogi dla grafu małego miasta:

In [167]:
start_time = time.perf_counter()
start_city = "21"
road, distance = distance_of_travelling_salesman(small_pathways, start_city)
end_time = time.perf_counter()
total_time = end_time - start_time
print(f"Cities that were visited from {start_city} are: ")
print(road)
print(f"and the distance is {distance}")
print(f"It took programme {total_time} for solving travelling salesman problem")

Cities that were visited from 21 are: 
['21', '39', '1', '34', '25', '42', '11', '4', '33', '44', '13', '6', '22', '23', '47', '29', '5', '8', '45', '28', '17', '40', '24', '37', '46', '38', '43', '48', '30', '19', '14', '18', '10', '15', '32', '2', '27', '7', '31', '49', '12', '36', '21']
and the distance is 631.46
It took programme 0.040660290978848934 for solving travelling salesman problem


Dla grafu o niewielkiej liczbie wierzchołków
zaimplementowana heurystyka działa błyskawicznie, zwracając
wynik w zaledwie około 0.04 sekundy. Algorytm prawidłowo
wyznaczył cykl komiwojażera. Co najważniejsze, dzięki 
wykorzystaniu Dijkstry pod spodem, z sukcesem ominął problem
"ślepych zaułków" i odizolowanych części grafu, znajdując
legalną trasę omijającą luki w strukturze miejskiej.

### [4.2] Średnie Miasto

In [168]:
start_time = time.perf_counter()
start_city = "70"
road, distance = distance_of_travelling_salesman(mid_pathways, start_city)
end_time = time.perf_counter()
total_time = end_time - start_time
print(f"Cities that were visited from {start_city} are: ")
print(road)
print(f"and the distance is {distance}")
print(f"It took programme {total_time} for solving travelling salesman problem")

Cities that were visited from 70 are: 
['70', '8', '129', '61', '120', '56', '110', '102', '36', '39', '198', '62', '83', '131', '165', '65', '46', '148', '132', '119', '63', '99', '78', '125', '14', '12', '10', '95', '34', '85', '124', '75', '156', '117', '123', '19', '73', '88', '135', '106', '101', '25', '37', '186', '184', '55', '193', '9', '185', '92', '81', '109', '121', '126', '27', '49', '71', '7', '67', '58', '42', '199', '80', '190', '160', '192', '69', '38', '87', '33', '166', '162', '151', '97', '182', '64', '104', '174', '3', '179', '18', '136', '159', '53', '32', '21', '51', '66', '112', '82', '4', '158', '45', '171', '133', '175', '74', '168', '94', '79', '134', '54', '59', '20', '76', '157', '0', '114', '57', '30', '137', '191', '188', '161', '194', '89', '15', '72', '128', '180', '170', '149', '183', '5', '100', '167', '130', '178', '146', '196', '16', '43', '111', '107', '103', '155', '187', '98', '50', '172', '195', '90', '93', '164', '173', '28', '96', '181', '105',

### [4.3] Duże miasto

In [169]:
start_time = time.perf_counter()
start_city = "500"
road, distance = distance_of_travelling_salesman(big_pathways, start_city)
end_time = time.perf_counter()
total_time = end_time - start_time
print(f"Cities that were visited from {start_city} are: ")
print(road)
print(f"and the distance is {distance}")
print(f"It took programme {total_time} for solving travelling salesman problem")

KeyboardInterrupt: 

Eksperyment dla dużego grafu (1000 węzłów, ponad 20 000 krawędzi) 
zakończył się klasyczną "eksplozją kombinatoryczną". O ile prosta heurystyka 
z pod-wywołaniami Dijkstry świetnie sprawdziła się dla 200 miast, tak przy 
grafie rzędu N=1000 ilość wymaganych operacji rośnie do miliardów. 
Każdy krok komiwojażera wymusza weryfikację setek nieodwiedzonych sąsiadów, 
z których każdy odpala pełną Dijkstrę na potężnej siatce 20 tysięcy dróg.